In [0]:
%pip install python-dotenv azure-identity azure-storage-file-datalake --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
from dotenv import load_dotenv, find_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# 1. CONFIGURAÇÃO

load_dotenv(find_dotenv(), override=True)

CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
TENANT_ID = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")
ACCOUNT = os.getenv("STORAGE_ACCOUNT_NAME")
RAW_CONTAINER = os.getenv("ADLS_CONTAINER", "raw")
BRONZE_CONTAINER = os.getenv("ADLS_BRONZE_CONTAINER", RAW_CONTAINER)
SILVER_CONTAINER = os.getenv("ADLS_SILVER_CONTAINER", BRONZE_CONTAINER)

if not all([CLIENT_ID, TENANT_ID, CLIENT_SECRET, ACCOUNT]):
    raise ValueError("Existem variáveis pendentes no .env!")

host = f"{ACCOUNT}.dfs.core.windows.net"
spark_auth = {
    f"fs.azure.account.auth.type.{host}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{host}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{host}": CLIENT_ID,
    f"fs.azure.account.oauth2.client.secret.{host}": CLIENT_SECRET,
    f"fs.azure.account.oauth2.client.endpoint.{host}": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

# Cliente do SDK do Azure: usado só para listar pastas/arquivos (não depende do spark.conf)
adls = DataLakeServiceClient(
    account_url=f"https://{host}",
    credential=ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET),
)

def abfss(container, rel):
    return f"abfss://{container}@{host}/{rel.strip('/')}"

BRONZE_PATH = abfss(BRONZE_CONTAINER, "bronze/physical_vendas_caixa")
SILVER_REL = "silver/physical_vendas_caixa"
SILVER_PATH = abfss(SILVER_CONTAINER, SILVER_REL)
REJEITADOS_PATH = abfss(SILVER_CONTAINER, "silver/physical_vendas_caixa_rejeitados")

# Regra 2 (FK de loja): lê physical_lojas direto da RAW.
# LOJAS_REL (opcional no .env) = caminho DENTRO do container raw, ex.: LOJAS_REL=physical_lojas
# Se não for informado, o notebook procura no container raw algo com "physical_lojas" no nome.
LOJAS_REL = os.getenv("LOJAS_REL")

TIPOS_PAGAMENTO_VALIDOS = ["Cartão", "Dinheiro", "Pix"]
UUID_REGEX = r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$"
PASTAS_IGNORADAS = ("bronze", "silver", "gold")

def listar(container, rel="", recursivo=False):
    """Lista caminhos (relativos ao container) via SDK. Retorna [] se não existir."""
    fs = adls.get_file_system_client(container)
    try:
        return [(p.name, p.is_directory) for p in fs.get_paths(path=rel or None, recursive=recursivo)]
    except Exception:
        return []

def ler_delta(path, nome):
    try:
        return spark.read.format("delta").options(**spark_auth).load(path)
    except Exception as e:
        raise RuntimeError(f"Não foi possível ler {nome} em {path}. Detalhe: {e}")

def gravar_delta(df, path, particoes=None):
    w = (df.write.format("delta").mode("overwrite")
         .option("overwriteSchema", "true").options(**spark_auth))
    if particoes:
        w = w.partitionBy(*particoes)
    w.save(path)

In [0]:

# 2. LEITURA

df_bronze = ler_delta(BRONZE_PATH, "a Bronze de vendas")
total_bronze = df_bronze.count()
print(f"Linhas na Bronze: {total_bronze}")

def localizar_lojas():
    if LOJAS_REL:
        return LOJAS_REL.strip("/")
    candidatos = [
        nome for nome, _ in listar(RAW_CONTAINER, recursivo=True)
        if nome.split("/")[0].lower() not in PASTAS_IGNORADAS and "_delta_log" not in nome
    ]
    for termo in ("physical_lojas", "lojas"):
        achados = [n for n in candidatos if termo in n.split("/")[-1].lower()]
        if achados:
            return min(achados, key=len)  # o mais "raso" (pasta antes dos arquivos dela)
    return None

def ler_auto(container, rel):
    """Lê Delta, Parquet, JSON ou CSV detectando o formato pelos arquivos do caminho."""
    path = abfss(container, rel)
    itens = listar(container, rel, recursivo=True)
    arquivos = [n for n, is_dir in itens if not is_dir] or [rel]  # rel pode ser um arquivo único
    if any("_delta_log" in n for n in arquivos):
        return spark.read.format("delta").options(**spark_auth).load(path), "delta"
    exts = {n.rsplit(".", 1)[-1].lower() for n in arquivos if "." in n.split("/")[-1]}
    leitor = spark.read.options(**spark_auth)
    if "parquet" in exts:
        return leitor.parquet(path), "parquet"
    if "json" in exts:
        return leitor.option("multiLine", "true").json(path), "json"
    if exts & {"csv", "txt"}:
        for sep in [",", ";", "|"]:
            df_csv = leitor.option("header", "true").option("sep", sep).option("encoding", "UTF-8").csv(path)
            if len(df_csv.columns) > 1:
                return df_csv, f"csv (sep '{sep}')"
        return df_csv, "csv"
    raise RuntimeError(f"Formato não reconhecido em {path}. Extensões encontradas: {exts or 'nenhuma'}")

lojas_rel = localizar_lojas()
aplicar_regra_2 = lojas_rel is not None
if aplicar_regra_2:
    df_lojas, formato = ler_auto(RAW_CONTAINER, lojas_rel)
    if "id_loja" not in df_lojas.columns:
        raise RuntimeError(f"A tabela de lojas em {lojas_rel} não tem a coluna id_loja. Colunas: {df_lojas.columns}")
    # compara como texto sem espaços, para não falhar por diferença de tipo (int x string)
    lojas_validas = (
        df_lojas.select(F.trim(F.col("id_loja").cast("string")).alias("_id_loja_fk"))
        .where(F.col("_id_loja_fk").isNotNull() & (F.col("_id_loja_fk") != ""))
        .distinct()
    )
    print(f"Regra 2 ATIVA: {lojas_validas.count()} lojas lidas de {RAW_CONTAINER}/{lojas_rel} ({formato})")
else:
    print("AVISO: Regra 2 (FK id_loja -> physical_lojas) NÃO aplicada: tabela de lojas não encontrada na raw. "
          "Defina LOJAS_REL no .env.")

Linhas na Bronze: 375000
Regra 2 ATIVA: 29 lojas lidas de raw/batch-data/physical_lojas.csv (csv (sep ','))


In [0]:

# 3. PADRONIZAÇÃO NECESSÁRIA PARA AS REGRAS

# Regra 4: compara tipo_pagamento sem acento/caixa/espaços e devolve o valor oficial do PDV.

pgto_normalizado = F.translate(
    F.lower(F.trim(F.col("tipo_pagamento"))), "ãáàâäéêèíìóôõòúùüç", "aaaaaeeeiiooooouuuc"
)

df = (
    df_bronze
    .withColumn("id_transacao", F.lower(F.trim(F.col("id_transacao"))))
    .withColumn("valor_total_venda", F.round(F.col("valor_total_venda"), 2))
    .withColumn(
        "tipo_pagamento",
        F.when(pgto_normalizado == "cartao", "Cartão")
         .when(pgto_normalizado == "dinheiro", "Dinheiro")
         .when(pgto_normalizado == "pix", "Pix")
    )
    # Regra 5: colunas de partição a partir de dt_venda
    .withColumn("ano", F.year("dt_venda"))
    .withColumn("mes", F.month("dt_venda"))
)

if aplicar_regra_2:
    df = (
        df.join(F.broadcast(lojas_validas),
                F.trim(df["id_loja"].cast("string")) == lojas_validas["_id_loja_fk"], "left")
          .withColumn("_loja_existe", F.col("_id_loja_fk").isNotNull())
          .drop("_id_loja_fk")
    )
else:
    df = df.withColumn("_loja_existe", F.lit(True))

In [0]:

# 4. REGRAS TÉCNICAS 1 A 4
# Cada registro recebe TODOS os motivos que violou.

id_vazio = F.col("id_transacao").isNull() | (F.col("id_transacao") == "")

motivos = [
    F.when(id_vazio, F.lit("R1: id_transacao nulo")),
    F.when(~id_vazio & ~F.col("id_transacao").rlike(UUID_REGEX), F.lit("R1: id_transacao não é UUID")),
    F.when(~F.col("_loja_existe"), F.lit("R2: id_loja não existe em physical_lojas")),
    F.when(F.col("valor_total_venda").isNull() | (F.col("valor_total_venda") <= 0),
           F.lit("R3: valor_total_venda <= 0 ou nulo")),
    F.when(F.col("tipo_pagamento").isNull(), F.lit("R4: tipo_pagamento fora de Cartão/Dinheiro/Pix")),
    F.when(F.col("dt_venda").isNull(), F.lit("R5: dt_venda nula (sem partição)")),
]

df = df.withColumn("motivo_rejeicao", F.concat_ws(" | ", *motivos))
df = df.withColumn("motivo_rejeicao", F.when(F.col("motivo_rejeicao") != "", F.col("motivo_rejeicao")))

df_invalidos = df.filter(F.col("motivo_rejeicao").isNotNull())
df_validos = df.filter(F.col("motivo_rejeicao").isNull())

# Regra 1 (duplicidade): mantém o registro ingerido mais recentemente; as cópias vão para a quarentena.
janela = Window.partitionBy("id_transacao").orderBy(F.col("bronze_ingested_at").desc())
df_validos = df_validos.withColumn("_rn", F.row_number().over(janela))

df_duplicados = (
    df_validos.filter(F.col("_rn") > 1)
    .withColumn("motivo_rejeicao", F.lit("R1: id_transacao duplicado (cópia mais antiga)"))
    .drop("_rn")
)
df_validos = df_validos.filter(F.col("_rn") == 1).drop("_rn")

In [0]:

# 5. GRAVAÇÃO

df_silver = (
    df_validos
    .drop("_loja_existe", "motivo_rejeicao")
    .withColumn("silver_processed_at", F.current_timestamp())
)

# Regra 5: Silver particionada por ano e mês
gravar_delta(df_silver, SILVER_PATH, particoes=["ano", "mes"])

df_rejeitados = (
    df_invalidos.unionByName(df_duplicados)
    .drop("_loja_existe")
    .withColumn("silver_processed_at", F.current_timestamp())
)
gravar_delta(df_rejeitados, REJEITADOS_PATH)

In [0]:

# 6. VALIDAÇÃO

df_check = ler_delta(SILVER_PATH, "a Silver")
df_rej = ler_delta(REJEITADOS_PATH, "a quarentena")

total_silver = df_check.count()
total_rej = df_rej.count()

print(f"Bronze: {total_bronze} | Silver: {total_silver} | Rejeitados: {total_rej} | "
      f"Conferência: {'OK' if total_silver + total_rej == total_bronze else 'DIVERGENTE'}")

checagens = {
    "R1 nulos": df_check.filter(F.col("id_transacao").isNull()).count(),
    "R1 duplicados": df_check.groupBy("id_transacao").count().filter("count > 1").count(),
    "R3 valor <= 0": df_check.filter(F.col("valor_total_venda") <= 0).count(),
    "R4 pagamento inválido": df_check.filter(~F.col("tipo_pagamento").isin(TIPOS_PAGAMENTO_VALIDOS)).count(),
}
print("\nChecagens na Silver (todas devem ser 0):")
for regra, qtd in checagens.items():
    print(f"  {regra}: {qtd}")
print(f"  R2 FK de loja: {'aplicada' if aplicar_regra_2 else 'PENDENTE (tabela de lojas não encontrada)'}")

# Regra 5: confirma o particionamento olhando as pastas gravadas (ano=AAAA/mes=M)
pastas_ano = sorted(n.split("/")[-1] for n, d in listar(SILVER_CONTAINER, SILVER_REL) if d and "ano=" in n)
print(f"  R5 partições de ano gravadas: {pastas_ano}")

print("\nRejeitados por motivo:")
display(df_rej.groupBy("motivo_rejeicao").count().orderBy(F.desc("count")))

df_check.printSchema()
display(df_check.limit(100))

Bronze: 375000 | Silver: 375000 | Rejeitados: 0 | Conferência: OK

Checagens na Silver (todas devem ser 0):
  R1 nulos: 0
  R1 duplicados: 0
  R3 valor <= 0: 0
  R4 pagamento inválido: 0
  R2 FK de loja: aplicada
  R5 partições de ano gravadas: ['ano=2024', 'ano=2025', 'ano=2026']

Rejeitados por motivo:


motivo_rejeicao,count


root
 |-- id_transacao: string (nullable = true)
 |-- id_loja: integer (nullable = true)
 |-- id_caixa: integer (nullable = true)
 |-- id_operador: integer (nullable = true)
 |-- dt_venda: timestamp (nullable = true)
 |-- valor_total_venda: double (nullable = true)
 |-- cpf_cliente: string (nullable = true)
 |-- tipo_pagamento: string (nullable = true)
 |-- bronze_source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- silver_processed_at: timestamp (nullable = true)



id_transacao,id_loja,id_caixa,id_operador,dt_venda,valor_total_venda,cpf_cliente,tipo_pagamento,bronze_source_file,bronze_ingested_at,ano,mes,silver_processed_at
00608bc2-9e1e-4881-b6cf-cb47b5643dbe,10,3,10,2026-05-14T20:04:57.000Z,340.8,null,Dinheiro,abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_vendas_caixa.csv,2026-10-15T01:21:31.492Z,2026,5,2026-10-15T17:36:54.844Z
0067c419-7dbe-4177-abc1-68150db5246c,24,1,15,2026-05-18T23:14:43.000Z,186.96,null,Dinheiro,abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_vendas_caixa.csv,2026-10-15T01:21:31.492Z,2026,5,2026-10-15T17:36:54.844Z
00a12b39-8bd4-41ef-88c6-b682add418e0,6,2,16,2026-05-28T06:05:57.000Z,125.26,null,Cartão,abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_vendas_caixa.csv,2026-10-15T01:21:31.492Z,2026,5,2026-10-15T17:36:54.844Z
00aa91e0-3688-421f-af6e-d507284209a7,30,4,18,2026-05-06T22:16:27.000Z,225.85,null,Pix,abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_vendas_caixa.csv,2026-10-15T01:21:31.492Z,2026,5,2026-10-15T17:36:54.844Z
011b49cf-f192-45a6-9d07-20cdf1023d22,10,3,18,2026-05-28T13:34:06.000Z,455.03,null,Cartão,abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_vendas_caixa.csv,2026-10-15T01:21:31.492Z,2026,5,2026-10-15T17:36:54.844Z
014f2fc8-21e0-4c97-a298-5b44c6ce721f,16,2,17,2026-05-06T22:37:30.000Z,570.12,null,Cartão,abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_vendas_caixa.csv,2026-10-15T01:21:31.492Z,2026,5,2026-10-15T17:36:54.844Z
0150ba74-126a-4e5b-8bd9-1d9d8db329a7,11,5,9,2026-05-06T09:44:46.000Z,285.24,null,Cartão,abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_vendas_caixa.csv,2026-10-15T01:21:31.492Z,2026,5,2026-10-15T17:36:54.844Z
01ac549a-7139-42ea-95d3-2e7297c286b7,26,3,1,2026-05-26T07:28:25.000Z,80.79,54732689191,Cartão,abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_vendas_caixa.csv,2026-10-15T01:21:31.492Z,2026,5,2026-10-15T17:36:54.844Z
02cbd4d8-4e62-46af-aba3-f6f9b3a2a64f,3,1,19,2026-05-27T18:40:20.000Z,24.75,56031279452,Cartão,abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_vendas_caixa.csv,2026-10-15T01:21:31.492Z,2026,5,2026-10-15T17:36:54.844Z
03005294-3208-4a05-ae52-63cf25f09d4d,9,3,13,2026-05-11T22:01:59.000Z,69.13,02941563725,Pix,abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_vendas_caixa.csv,2026-10-15T01:21:31.492Z,2026,5,2026-10-15T17:36:54.844Z
